# Stage 05 · Model Evaluation

Load the Stage 03 test split and the Stage 04 model, calculate RMSE, MAE, and R², then log the evaluation to MLflow.

In [ ]:
import os
import sys
from pathlib import Path

if Path.cwd().name == "research":
    os.chdir("..")

sys.path.insert(0, str(Path.cwd()))
Path.cwd()

## Configuration entity

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    metrics_path: Path
    target_column: str
    tracking_uri: str
    experiment_name: str
    model_params: dict
    registered_model_name: str

## Configuration manager

In [ ]:
from src.forcastingproject.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH, SCHEMA_FILE_PATH
from src.forcastingproject.utils.common import create_directories, read_yaml_file


class ConfigurationManager:
    def __init__(
        self,
        config_file_path: Path = CONFIG_FILE_PATH,
        params_file_path: Path = PARAMS_FILE_PATH,
        schema_file_path: Path = SCHEMA_FILE_PATH
    ):
        self.config = read_yaml_file(config_file_path)
        self.params = read_yaml_file(params_file_path)
        self.schema = read_yaml_file(schema_file_path)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=Path(config.root_dir),
            test_data_path=Path(config.test_data_path),
            model_path=Path(config.model_path),
            metrics_path=Path(config.metrics_path),
            target_column=self.schema.TARGET_COLUMN.name,
            tracking_uri=os.environ.get("MLFLOW_TRACKING_URI") or config.tracking_uri,
            experiment_name=config.experiment_name,
            model_params=dict(self.params.model_trainer),
            registered_model_name=config.registered_model_name,
        )
        return model_evaluation_config

## Model evaluation component

In [ ]:
import joblib
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from src.forcastingproject.logging.logger import logger
from src.forcastingproject.utils.common import save_json


class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def initiate_model_evaluation(self, *, local_only: bool = False) -> dict[str, float]:
        test_data = pd.read_csv(self.config.test_data_path)
        features = test_data.drop(columns=self.config.target_column)
        target = test_data[self.config.target_column]
        model = joblib.load(self.config.model_path)
        predictions = model.predict(features)

        metrics = {
            "rmse": float(mean_squared_error(target, predictions) ** 0.5),
            "mae": float(mean_absolute_error(target, predictions)),
            "r2": float(r2_score(target, predictions)),
        }
        save_json(self.config.metrics_path, metrics)
        logger.info(f"Saved evaluation metrics to {self.config.metrics_path}")

        if not local_only:
            self.log_to_mlflow(model, features.head(), predictions[:5], metrics)

        return metrics

    def log_to_mlflow(self, model, input_example, predictions, metrics):
        import mlflow
        import mlflow.sklearn
        from mlflow.models import infer_signature

        mlflow.set_tracking_uri(self.config.tracking_uri)
        mlflow.set_experiment(self.config.experiment_name)

        with mlflow.start_run(run_name="random-forest-evaluation") as run:
            mlflow.log_params(self.config.model_params)
            mlflow.log_metrics(metrics)
            mlflow.log_artifact(str(self.config.metrics_path))
            mlflow.sklearn.log_model(
                sk_model=model,
                name="model",
                input_example=input_example,
                signature=infer_signature(input_example, predictions),
                registered_model_name=self.config.registered_model_name,
            )
            logger.info(f"Logged evaluation to MLflow run {run.info.run_id}")

## Run evaluation

Set `local_only=False` to log to DagsHub. The notebook kernel reads `MLFLOW_TRACKING_USERNAME` and `MLFLOW_TRACKING_PASSWORD` from its environment.

In [ ]:
config = ConfigurationManager()
model_evaluation_config = config.get_model_evaluation_config()
model_evaluation = ModelEvaluation(config=model_evaluation_config)
model_evaluation.initiate_model_evaluation(local_only=True)

In [ ]:
import json

json.loads(model_evaluation_config.metrics_path.read_text())